# 13 · Explore: profiles, masks and composites in one-liners

A playground. Every cell is short and stands on the CONFIG cell only, so copy, change and recombine freely.

| What | Call | Gives |
|---|---|---|
| SAM's own hourly STAT profiles (instant) | `S = stat_profiles(EXPERIMENT, RV_LIST)` | Dataset `(rv, z)`: `MSE`, `MSECLD` (cloud MSE), `CLD`, `TABS`, `QV`, `RELH`, … (MSE in **K**: ×1.004 → kJ/kg) |
| Composites with any mask (3D output; minutes first time, then cached) | `C = load(EXPERIMENT, RV_LIST, masks.cloud)` | Dataset `(rv, group, z)`: every 3D field + `MSE`, `MSE_frozen`, `THETA_E` and saturated versions; coords `RH`, `fraction`, `fraction_z` |
| One panel | `plot_profiles(C, "MSE", group="cloud")` / `plot_profiles(C, "MSE", by="group", rv=0)` | Lines per r_v (coloured by RH) or per group; `anomaly=True`, `xlim=`, `zmax=`, `sat=False` |
| Many panels | `plot_grid(C, ["W", "QN", "MSE"], group="cloud")` | Same options |
| Stability | `stability_profiles(C)` | `N2`, `N2_m`, `N2_m_entropy`, `N2_m_loading` (Emanuel 6.1.7 / 6.2.10) |

Masks come in two kinds: **column** masks (the same columns at every height, e.g. near-surface W) and **level** masks (chosen at each height, e.g. *cloudy here*). The library ones are in `samheat/analysis/masks.py`; write your own in section 7.

In [ ]:
# RUN THIS CELL FIRST: lets this notebook find the samheat code in this repo (no install needed)
import os, sys, pathlib
def _find_repo():
    starts = [pathlib.Path(os.environ.get("JPY_SESSION_NAME", ".")).resolve(), pathlib.Path.cwd().resolve()]
    for s in starts:
        for d in [s, *s.parents]:
            if (d / "samheat" / "__init__.py").exists():
                return d
    for d in [pathlib.Path("/scratch/ttg2015/Torch/ExtremeHeat/SAM_HEAT"), pathlib.Path.home() / "SAM_HEAT"]:
        if (d / "samheat" / "__init__.py").exists():
            return d
    raise FileNotFoundError("Could not find the SAM_HEAT folder. Set REPO by hand, e.g. REPO = pathlib.Path('/scratch/.../SAM_HEAT')")
REPO = _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import samheat
print("using samheat from", pathlib.Path(samheat.__file__).parent)

In [ ]:
# ---- CONFIG ----
import matplotlib.pyplot as plt
from samheat.paths import SITE
from samheat.analysis.explore import load, stat_profiles, plot_profiles, plot_grid, plot_cloud_env, masks
from samheat.analysis.stability import stability_profiles

EXPERIMENT = "vdd2025_vddgrid"          # or "vdd2025_repro", "vdd2025_vddgrid_U5", ...
RV_LIST = [0, 200, 500, 1000, 2000]
TMIN, TMAX = 30, 60                     # days of Stage 2

In [ ]:
# ---- What masks exist? ----
for kind, group in (("column", masks.COLUMN_MASKS), ("level", masks.LEVEL_MASKS)):
    for name, fn in group.items():
        print(f"{kind:6s}  masks.{name:15s} {fn.__doc__.strip().splitlines()[0]}")

## 1. SAM's own STAT profiles (no 3D reading: instant)
`MSECLD` is SAM's cloud-conditional MSE (cloud = liquid + ice > min(0.01 g/kg, 1 % of saturation)), written every hour. SAM writes MSE in K (h/c_p).

In [ ]:
S = stat_profiles(EXPERIMENT, RV_LIST, tmin=TMIN, tmax=TMAX)
S["MSE_kJ"] = 1.004 * S.MSE                 # K -> kJ/kg
S["MSECLD_kJ"] = 1.004 * S.MSECLD.where(S.CLD > 0)   # cloud MSE only where there is cloud
print([v for v in S.data_vars][:40], "...")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
plot_profiles(S, "MSE_kJ", ax=ax[0], xlim=(320, 350))
plot_profiles(S, "MSECLD_kJ", ax=ax[1], xlim=(320, 350), legend=False)
plot_profiles(S, "CLD", ax=ax[2], legend=False)
plt.show()

## 2. Composites from the 3D output
The first call per mask reads every snapshot (minutes per run); after that it loads from the cache in seconds.

In [ ]:
C_cloud = load(EXPERIMENT, RV_LIST, masks.cloud, tmin=TMIN, tmax=TMAX)                  # level mask: cloudy vs clear
C_cloud

In [ ]:
C_w = load(EXPERIMENT, RV_LIST, masks.symmetric, k=1, X=0.3, tmin=TMIN, tmax=TMAX)      # column mask: near-surface W

## 3. Quick plots

In [ ]:
plot_profiles(C_cloud, "MSE", group="cloud"); plt.show()              # cloud MSE per r_v (dashed: saturated MSE*)

In [ ]:
plot_profiles(C_cloud, "MSE", by="group", rv=RV_LIST[0]); plt.show()  # cloud vs clear vs domain mean, one run

In [ ]:
plot_grid(C_cloud, ["QN", "W", "TABS", "QV", "MSE", "MSE_frozen", "THETA_E", "QP"], group="cloud"); plt.show()

In [ ]:
plot_profiles(C_cloud, "TABS", group="cloud", anomaly=True); plt.show()   # cloud minus domain mean

In [ ]:
plot_profiles(C_cloud, "fraction_z", group="cloud", xlim="auto"); plt.show()   # fraction of points that are cloudy, per level

## 4. Reproduce SAM's own cloud statistics from the 3D output
`masks.cloud` uses SAM's exact criterion (`statistics.f90`): **QN > min(10⁻⁵ kg/kg, 0.01·q*_water(T̄(z), p̄(z)))**, and `MSE_SAM` is SAM's MSE, (c_pT + gz + L_c q_v)/1000 in kJ/kg, i.e. 1.004 × the STAT `MSE`/`MSECLD` (in K). The 3D composite should therefore reproduce STAT `CLD` and `MSECLD` up to sampling: 3-hourly snapshots vs every statistics sample in the hour. (On synthetic data the match is exact; see `tests/test_explore.py`.)

In [ ]:
from samheat.analysis.explore import compare_cloud_with_stat, rh_colors
display(compare_cloud_with_stat(C_cloud, S, zmax=12000))

col = rh_colors(C_cloud)
fig, ax = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for rv in C_cloud.rv.values:
    ax[0].plot(C_cloud.fraction_z.sel(rv=rv, group="cloud"), C_cloud.z / 1000, c=col[rv], label=f"rv {rv:g} 3D")
    ax[0].plot(S.CLD.sel(rv=rv), S.z / 1000, c=col[rv], ls=":", label=f"rv {rv:g} STAT")
    ax[1].plot(C_cloud.MSE_SAM.sel(rv=rv, group="cloud"), C_cloud.z / 1000, c=col[rv])
    ax[1].plot(S.MSECLD_kJ.sel(rv=rv), S.z / 1000, c=col[rv], ls=":")
ax[0].set_xlabel("cloud fraction"); ax[1].set_xlabel("cloud MSE, SAM definition (kJ/kg)"); ax[1].set_xlim(320, 350)
ax[0].set_ylabel("z (km)"); ax[0].set_ylim(0, 16); ax[0].legend(fontsize=7); fig.suptitle("solid: from 3D output, dotted: SAM's STAT file")
plt.show()

## 4b. Cloud, total and environmental saturated MSE
Solid: MSE of cloudy air. Dashed: domain mean. Dotted: saturated MSE h* of the clear-air environment. Colour: near-surface RH.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 6), sharey=True)
plot_cloud_env(C_cloud, "MSE", ax=ax[0])                 # your simple MSE
plot_cloud_env(C_cloud, "MSE_SAM", ax=ax[1])             # SAM's MSE (same as the STAT file)
plt.show()

## 4c. Precipitation definitions
| Mask | Kind | Definition |
|---|---|---|
| `masks.rain` | level | QP > 0.1 g/kg (= 10⁻⁴ kg/kg, SAM's only precipitation threshold) |
| `masks.hydrometeors` | level | SAM's exclusive HYDRO classes: cloud, else rain-only (QP > 0.1), else clear. Rain-only below cloud base = where rain re-evaporates |
| `masks.precipitating_cloud` | level | cloudy and QP > 0.1 vs non-precipitating cloud vs clear |
| `masks.raining_column` | column | max QP below 1 km > 0.1 g/kg (3D proxy for surface rain; 0.1 g/kg ≈ a few mm/h) |
| `masks.heavy_rain_column` | column | top q % of near-surface QP in each snapshot (proxy for precipitation-extreme columns) |
| `masks.cloudy_column` | column | cloud water path > 0.02 kg/m² (SAM's column-cloud threshold) |
Rate-based definitions (surface rain > 0.1 or 1 mm/h; convective/stratiform) need the 2D `Prec` field; ask when you want them. Full survey with citations: `exports/cloud_precip_definitions.md`.

In [ ]:
C_hyd = load(EXPERIMENT, RV_LIST, masks.hydrometeors, tmin=TMIN, tmax=TMAX)
fig, ax = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for a, g in zip(ax, ["cloud", "rain_only", "clear"]):
    plot_profiles(C_hyd, "fraction_z", group=g, ax=a, legend=(g == "cloud"))
    a.set_title(f"fraction of points: {g}")
plt.show()
plot_profiles(C_hyd, "QV", by="group", rv=RV_LIST[-1], anomaly=True); plt.show()   # e.g. moistening where rain evaporates

## 4d. 3D clouds: white boxes (and rain in blue)
`renderer="pyvista"` is fast and pretty (about 0.1–0.5 s per frame); it needs `pip install pyvista imageio imageio-ffmpeg` in your Jupyter environment and, on a cluster without a display, EGL or OSMesa (VTK ≥ 9.4 picks one itself). If PyVista fails, use `renderer="matplotlib"` (no extra packages, ~5–10 s per frame). Heights are exaggerated ×3 by default (`zscale`). Notes: `exports/cloud_3d_visualization.md`.

In [ ]:
# %pip install pyvista imageio imageio-ffmpeg        # once, then restart the kernel
from IPython.display import Image, Video
from samheat.analysis.clouds3d import cloud_frame, cloud_movie
RUN_DIR = SITE.work_root / EXPERIMENT / f"stage2_rv{RV_LIST[0]:g}"
png = cloud_frame(RUN_DIR, time=45.0, out="clouds_day45.png", renderer="auto")
Image(str(png))

In [ ]:
# the movie: every snapshot between TMIN and TMAX, camera rotating slowly (minutes with pyvista)
mp4 = cloud_movie(RUN_DIR, out=f"clouds_rv{RV_LIST[0]:g}.mp4", tmin=TMIN, tmax=TMAX, every=1, fps=12, rotate=0.5)
Video(str(mp4), embed=False)

## 5. Stability of each group (Emanuel 6.1.7 / 6.2.10)

In [ ]:
ST = stability_profiles(C_cloud)
plot_grid(ST, ["N2", "N2_m", "N2_m_entropy", "N2_m_loading"], group="clear"); plt.show()

## 6. Do it yourself with xarray
Everything above is a plain xarray Dataset: `.sel`, arithmetic, `.mean`, `.plot` all work.

In [ ]:
diff = C_cloud.MSE.sel(group="cloud") - C_cloud.MSE.sel(group="clear")      # (rv, z): cloud minus clear-air MSE
diff.plot(y="z", hue="rv"); plt.ylim(0, 16000); plt.axvline(0, c="k", lw=.5); plt.show()
diff.sel(z=slice(1000, 5000)).mean("z").to_pandas()                          # one number per run

## 7. Your own mask
A mask gets one snapshot `f` (dict of 3D arrays `(z, y, x)` in SAM units, plus the derived fields) and the heights `z`. Return `{group: boolean array}`: `(y, x)` for columns or `(z, y, x)` for level by level. Helpers you can use: `masks.w_at_levels(f)`, `masks.buoyancy(f)`, `masks.parcel_cape(f, z)`, `masks.sam_cloud_threshold(f, z)`.

In [ ]:
def my_mask(f, z, w_min=2.0):
    """Strong cloudy updrafts at each level vs everything else."""
    strong = (f["QN"] > masks.sam_cloud_threshold(f, z)) & (masks.w_at_levels(f) > w_min)
    return dict(strong=strong, rest=~strong)

C_my = load(EXPERIMENT, RV_LIST, my_mask, w_min=2.0, tmin=TMIN, tmax=TMAX)
plot_profiles(C_my, "MSE", group="strong"); plt.show()